# Stage C 03h — c16 conditional-generation capability diagnostics

This notebook generates continuations from distinct held-out *E. coli* accessions at three temperatures, compares them with equal-length real continuations, runs transparent ORF diagnostics and Prodigal gene/intergenic predictions, and records all outputs in the exploratory study ledger. These are sequence-realism diagnostics, not functional validation.

In [ ]:
# USER CONFIGURATION
REPO_URL='https://github.com/Gonza10V/SeqTrainer.git'
GIT_REF='PIN_AFTER_COMMIT'
DRIVE_ROOT='/content/drive/MyDrive/SeqTrainerStageC'
SOURCE_RUN='c16_deep_adaptive_5m_paper_exact'
OUTPUT_NAME='generation_diagnostics_v1'
TAXONOMY_MANIFEST=f'{DRIVE_ROOT}/stage_c_dataset/manifests/accession_manifest.parquet'
PROMPTS=4; PROMPT_TOKENS=32; NEW_TOKENS=1024
TEMPERATURES='0.8,1.0,1.2'; TOP_K=128; TOP_P=0.95; SEED=20260781


In [ ]:
from pathlib import Path
from google.colab import drive
import json, shutil, subprocess, sys
mount=Path('/content/drive')
if not (mount/'MyDrive').is_dir(): drive.mount(str(mount),timeout_ms=120000)
repo=Path('/content/SeqTrainer')
if not repo.exists(): subprocess.run(['git','clone',REPO_URL,str(repo)],check=True)
subprocess.run(['git','-C',str(repo),'fetch','origin'],check=True)
subprocess.run(['git','-C',str(repo),'checkout',GIT_REF],check=True)
subprocess.run([sys.executable,'-m','pip','install','-e',f'{repo}[torch,bacteria-titan]'],check=True)
import torch
if not torch.cuda.is_available(): raise RuntimeError('Select a Colab GPU runtime; generation is much faster on GPU.')
if not shutil.which('prodigal'):
    subprocess.run(['apt-get','update'],check=True)
    subprocess.run(['apt-get','install','-y','prodigal'],check=True)
selection=json.loads((Path(DRIVE_ROOT)/'runs/c1_tokenizers_cpu/tokenizer_selection.json').read_text())
dataset=Path(DRIVE_ROOT)/'stage_c_dataset/ordered_streams'/selection['selected_tokenizer']
run_dir=Path(DRIVE_ROOT)/'runs'/SOURCE_RUN
checkpoint=run_dir/'latest.pt'; output_dir=run_dir/OUTPUT_NAME
PROTOCOL=repo/'studies/stage_c_ecoli_escherichia_paper_deep_memory_v2/protocol.json'
AMENDMENT=repo/'studies/stage_c_ecoli_escherichia_paper_deep_memory_v2/amendments/c16_generation_diagnostics_v1.json'
STUDY_ROOT=Path(DRIVE_ROOT)/'study/stage_c_ecoli_escherichia_paper_deep_memory_v2'
for required in (dataset,checkpoint,Path(TAXONOMY_MANIFEST),PROTOCOL,AMENDMENT):
    if not required.exists(): raise FileNotFoundError(required)
subprocess.run(['seqtrainer-titans-stage-c-study','initialize','--protocol',str(PROTOCOL),'--study-root',str(STUDY_ROOT)],check=True)
source=json.loads(AMENDMENT.read_text()); drive_amendment=STUDY_ROOT/'amendments'/f"{source['amendment_id']}.json"
if not drive_amendment.exists():
    subprocess.run(['seqtrainer-titans-stage-c-study','amend','--protocol',str(PROTOCOL),'--study-root',str(STUDY_ROOT),'--amendment-id',source['amendment_id'],'--rationale',source['rationale'],'--classification',source['classification'],'--expected-impact',source['expected_impact'],'--changes',json.dumps(source['changes'],sort_keys=True)],check=True)
print('GPU:',torch.cuda.get_device_name(0)); print('Output:',output_dir)


In [ ]:
def run_logged(log_dir,label,command):
    try:
        subprocess.run(['seqtrainer-titans-stage-c-colab-run','--run-dir',str(log_dir),'--label',label,'--repo',str(repo),'--',*command],check=True)
    except subprocess.CalledProcessError:
        for path in (log_dir/'FAILED.txt',log_dir/'logs'/f'{label}.log'):
            if path.exists(): print(path.read_text(errors='replace')[-16000:])
        raise
report=output_dir/'generation_evaluation.json'
if not report.exists():
    command=['seqtrainer-titans-stage-c-generate','--dataset-dir',str(dataset),'--taxonomy-manifest',TAXONOMY_MANIFEST,'--checkpoint',str(checkpoint),'--output-dir',str(output_dir),'--split','val','--species','Escherichia coli','--prompts',str(PROMPTS),'--prompt-tokens',str(PROMPT_TOKENS),'--new-tokens',str(NEW_TOKENS),'--temperatures',TEMPERATURES,'--top-k',str(TOP_K),'--top-p',str(TOP_P),'--seed',str(SEED),'--device','cuda','--memory-mode','adaptive','--prodigal',shutil.which('prodigal'),'--protocol',str(PROTOCOL),'--protocol-amendment',str(AMENDMENT),'--run-id','adaptive_exploration_5m_generation']
    run_logged(output_dir,'generation_c16',command)
else: print('Completed generation report exists; generation is skipped:',report)
marker=STUDY_ROOT/'record_markers/c16_generation_diagnostics_v1.json'
if not marker.exists():
    subprocess.run(['seqtrainer-titans-stage-c-study','record','--protocol',str(PROTOCOL),'--protocol-amendment',str(AMENDMENT),'--study-root',str(STUDY_ROOT),'--run-id','adaptive_exploration_5m_generation','--evidence-tier','exploratory','--artifact',str(output_dir)],check=True)
    marker.parent.mkdir(parents=True,exist_ok=True); marker.write_text(json.dumps({'run_id':'adaptive_exploration_5m_generation','artifact':str(output_dir)},indent=2)+'\n')
subprocess.run(['seqtrainer-titans-stage-c-study','verify','--protocol',str(PROTOCOL),'--study-root',str(STUDY_ROOT)],check=True)


In [ ]:
result=json.loads(report.read_text())
print((output_dir/'GENERATION_REPORT.md').read_text())
print('\n6-mer Jensen-Shannon divergence to held-out continuations:')
for policy,values in result['kmer_jsd_to_heldout_reference'].items(): print(policy,values['6'])
print('\nProdigal summary:')
print(json.dumps(result['prodigal'],indent=2,sort_keys=True))
print('\nReview:',output_dir/'generation_gc.svg')
print('Review:',output_dir/'generation_kmer_jsd.svg')
print('Generated FASTA:',output_dir/'generated_sequences.fasta')
print('Full machine-readable evidence:',report)
